# Bab 1 — Common Conventions and API Elements of scikit-learn

*scikit-learn Cookbook, Third Edition* — John Sukup (Packt, 2025)

| No | Topik | Inti |
|----|-------|------|
| 1 | Prinsip desain scikit-learn | konsistensi, inspeksi, tidak ada kelas "aneh", komposisi, default yang masuk akal |
| 2 | **Estimator** | objek yang belajar dari data lewat `fit()` |
| 3 | **Predictor** | estimator yang bisa `predict()` / `predict_proba()` / `score()` |
| 4 | **Transformer** | estimator yang mengubah data lewat `transform()` / `fit_transform()` |
| 5 | Hyperparameter vs parameter terlatih | `get_params()`, `set_params()`, atribut ber-underscore (`coef_`) |
| 6 | Format data & dataset bawaan | matriks `X` berbentuk `(n_samples, n_features)`, vektor `y`, `load_*`/`make_*` |
| 7 | **Pipeline** | merangkai transformer + estimator menjadi satu objek |
| 8 | Evaluasi dasar | `train_test_split`, metrik, `cross_val_score` |


## 1. Prinsip Desain scikit-learn

scikit-learn dibangun di atas beberapa prinsip (Buitinck dkk., 2013, *"API design for machine learning software"*):

1. **Konsistensi** — semua objek berbagi antarmuka yang sama dan terdokumentasi dengan seragam.
2. **Inspeksi** — semua hyperparameter yang kita berikan dan semua parameter hasil belajar dapat dibaca sebagai atribut publik.
3. **Tidak ada kelas khusus untuk data** — data cukup direpresentasikan sebagai array NumPy, matriks sparse SciPy, atau DataFrame pandas. Tidak perlu "objek dataset" khusus.
4. **Komposisi** — algoritma kompleks dibangun dari blok sederhana (misalnya `Pipeline`, `ColumnTransformer`).
5. **Default yang masuk akal** — setiap hyperparameter punya nilai bawaan yang wajar sehingga model bisa langsung dipakai.


In [1]:
import numpy as np
import pandas as pd
import sklearn
import warnings
warnings.filterwarnings('ignore')

print('scikit-learn:', sklearn.__version__)
print('numpy       :', np.__version__)
print('pandas      :', pd.__version__)

scikit-learn: 1.6.1
numpy       : 2.1.3
pandas      : 2.2.3


## 2. Estimator — Objek yang Belajar dari Data

### Teori
**Estimator** adalah objek apa pun yang mempelajari sesuatu dari data melalui method `fit()`. Secara matematis, pada *supervised learning* kita punya data latih

$$\mathcal{D} = \{(\mathbf{x}_i, y_i)\}_{i=1}^{n}, \qquad \mathbf{X} \in \mathbb{R}^{n \times p},\; \mathbf{y} \in \mathbb{R}^{n}$$

dan `fit()` mencari fungsi $\hat f$ dari keluarga model $\mathcal{F}$ yang meminimalkan sebuah fungsi kerugian (loss) $L$:

$$\hat f = \arg\min_{f \in \mathcal{F}} \; \frac{1}{n}\sum_{i=1}^{n} L\big(y_i, f(\mathbf{x}_i)\big)$$

Untuk regresi linear, $\mathcal{F}$ adalah semua fungsi $f(\mathbf{x}) = \mathbf{w}^\top\mathbf{x} + b$ dan $L$ adalah *squared error* $(y - f(\mathbf{x}))^2$. Solusinya dikenal sebagai **Ordinary Least Squares (OLS)**:

$$\hat{\mathbf{w}} = (\mathbf{X}^\top \mathbf{X})^{-1}\mathbf{X}^\top \mathbf{y}$$

Polanya selalu tiga langkah: **buat objek → `fit()` → pakai**.

In [2]:
from sklearn.linear_model import LinearRegression

# Data sederhana: 5 sampel, 1 fitur
X = np.array([[1], [2], [3], [4], [5]])
y = np.array([1.0, 2.0, 3.0, 3.5, 5.0])

# 1) buat estimator, 2) latih, 3) pakai
model = LinearRegression()
model.fit(X, y)

X_new = np.array([[6], [7]])
print('Prediksi untuk X = 6 dan 7:', model.predict(X_new))
print('Kemiringan (coef_)  :', model.coef_)
print('Intersep (intercept_):', model.intercept_)

Prediksi untuk X = 6 dan 7: [5.75 6.7 ]
Kemiringan (coef_)  : [0.95]
Intersep (intercept_): 0.04999999999999938


**Interpretasi.** Model mempelajari garis $\hat y = 0{,}95x + 0{,}05$. Untuk $x=6$ prediksinya $0{,}95\cdot 6 + 0{,}05 = 5{,}75$, sama persis dengan output `predict()`. Perhatikan bahwa nilai yang *dipelajari* (`coef_`, `intercept_`) diakhiri garis bawah `_` — ini konvensi scikit-learn untuk atribut yang baru ada **setelah** `fit()` dipanggil.

Kita bisa memverifikasi rumus OLS secara manual dengan NumPy:

In [3]:
# Verifikasi rumus OLS: w = (X^T X)^{-1} X^T y, dengan kolom 1 untuk intersep
X_aug = np.hstack([np.ones((X.shape[0], 1)), X])
w = np.linalg.inv(X_aug.T @ X_aug) @ X_aug.T @ y
print('Hasil manual  -> intersep = %.4f, kemiringan = %.4f' % (w[0], w[1]))
print('Hasil sklearn -> intersep = %.4f, kemiringan = %.4f' % (model.intercept_, model.coef_[0]))

Hasil manual  -> intersep = 0.0500, kemiringan = 0.9500
Hasil sklearn -> intersep = 0.0500, kemiringan = 0.9500


### Estimator untuk *unsupervised learning*: `fit_predict()`

Pada *unsupervised learning* tidak ada $\mathbf{y}$. Contohnya **K-Means**, yang membagi data menjadi $K$ kelompok dengan meminimalkan *within-cluster sum of squares*:

$$\min_{C_1,\dots,C_K} \sum_{k=1}^{K}\sum_{\mathbf{x}_i \in C_k} \lVert \mathbf{x}_i - \boldsymbol{\mu}_k \rVert^2$$

dengan $\boldsymbol{\mu}_k$ = rata-rata (centroid) cluster ke-$k$. Karena biasanya kita ingin label cluster untuk data yang sama dengan data latih, scikit-learn menyediakan jalan pintas `fit_predict()` = `fit()` lalu `predict()` pada data yang sama.

In [4]:
from sklearn.cluster import KMeans

X_1d = np.array([[1], [2], [3], [10], [11], [12]])
kmeans = KMeans(n_clusters=2, n_init=10, random_state=42)
labels = kmeans.fit_predict(X_1d)
print('Label cluster :', labels)
print('Centroid      :', kmeans.cluster_centers_.ravel())
print('Inertia (WCSS):', kmeans.inertia_)

Label cluster : [0 0 0 1 1 1]
Centroid      : [ 2. 11.]
Inertia (WCSS): 4.0


**Interpretasi.** Dua kelompok yang terlihat jelas (1–3 dan 10–12) terpisah dengan centroid 2 dan 11. `inertia_` = $\sum (x_i-\mu_k)^2 = (1+0+1)+(1+0+1) = 4$, yaitu nilai fungsi objektif K-Means di titik optimum.

## 3. Predictor — `predict()`, `predict_proba()`, dan `score()`

### Teori
Untuk klasifikasi, banyak model tidak hanya mengeluarkan label tetapi juga **estimasi probabilitas** $\hat P(Y = k \mid \mathbf{x})$. Pada regresi logistik biner:

$$\hat P(Y=1\mid\mathbf{x}) = \sigma(\mathbf{w}^\top\mathbf{x} + b) = \frac{1}{1 + e^{-(\mathbf{w}^\top\mathbf{x} + b)}}$$

`predict()` lalu memilih kelas dengan probabilitas terbesar: $\hat y = \arg\max_k \hat P(Y=k\mid\mathbf{x})$.

Method `score()` memberi metrik bawaan yang berbeda menurut jenis model:
- klasifikasi → **akurasi** $= \frac{1}{n}\sum_i \mathbb{1}[\hat y_i = y_i]$
- regresi → **koefisien determinasi** $R^2 = 1 - \frac{\sum_i (y_i-\hat y_i)^2}{\sum_i (y_i-\bar y)^2}$

In [5]:
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.2, random_state=2024, stratify=iris.target
)

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

print('Prediksi 5 sampel pertama :', clf.predict(X_test[:5]))
print('Label sebenarnya          :', y_test[:5])
print('\nProbabilitas tiap kelas (5 sampel pertama):')
print(pd.DataFrame(clf.predict_proba(X_test[:5]), columns=iris.target_names).round(3))
print('\nAkurasi (score) pada data uji: %.3f' % clf.score(X_test, y_test))

Prediksi 5 sampel pertama : [0 2 0 1 2]
Label sebenarnya          : [0 2 0 2 2]

Probabilitas tiap kelas (5 sampel pertama):
   setosa  versicolor  virginica
0   0.965       0.035      0.000
1   0.000       0.070      0.930
2   0.972       0.028      0.000
3   0.011       0.613      0.376
4   0.001       0.192      0.807

Akurasi (score) pada data uji: 0.967


**Interpretasi.** Setiap baris `predict_proba()` berjumlah 1 karena merupakan distribusi probabilitas atas 3 kelas, dan label hasil `predict()` adalah kolom dengan probabilitas terbesar. Sampel ke-4 (indeks 3) adalah contoh menarik: label sebenarnya *virginica*, tetapi model memberi probabilitas 0,616 untuk *versicolor* dan 0,373 untuk *virginica*, sehingga salah prediksi. Probabilitas yang tidak "yakin" seperti ini biasanya muncul pada sampel di dekat batas antarkelas. Akurasi `score()` = 0,967 berarti 29 dari 30 sampel uji benar — tepat satu kesalahan, yaitu sampel ini.

## 4. Transformer — `fit()`, `transform()`, `fit_transform()`

### Teori
**Transformer** adalah estimator yang *mengubah* data. `fit()` mempelajari statistik dari data latih, sedangkan `transform()` menerapkan perubahan tersebut. Contoh paling umum adalah **standardisasi (z-score)**:

$$z_{ij} = \frac{x_{ij} - \mu_j}{\sigma_j}$$

dengan $\mu_j$ dan $\sigma_j$ adalah rata-rata dan simpangan baku fitur $j$ **yang dihitung dari data latih saja**.

**Aturan penting (mencegah *data leakage*):**
- data latih → `fit_transform()`
- data uji / data baru → **hanya** `transform()` (memakai $\mu_j,\sigma_j$ dari data latih)

Jika kita memanggil `fit()` pada data uji, informasi dari data uji "bocor" ke proses pelatihan dan hasil evaluasi menjadi terlalu optimistis.

In [6]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # belajar mean & std dari data latih
X_test_scaled = scaler.transform(X_test)         # pakai mean & std yang sama

print('Mean per fitur (dipelajari)  :', scaler.mean_.round(3))
print('Std per fitur (dipelajari)   :', scaler.scale_.round(3))
print('Mean data latih setelah scale:', X_train_scaled.mean(axis=0).round(3))
print('Std data latih setelah scale :', X_train_scaled.std(axis=0).round(3))
print('Mean data uji setelah scale  :', X_test_scaled.mean(axis=0).round(3))

# inverse_transform mengembalikan ke skala asli
print('\nBisa dikembalikan ke skala asli:', np.allclose(scaler.inverse_transform(X_test_scaled), X_test))

Mean per fitur (dipelajari)  : [5.853 3.084 3.775 1.214]
Std per fitur (dipelajari)   : [0.804 0.426 1.743 0.764]
Mean data latih setelah scale: [-0.  0.  0. -0.]
Std data latih setelah scale : [1. 1. 1. 1.]
Mean data uji setelah scale  : [-0.057 -0.315 -0.049 -0.097]

Bisa dikembalikan ke skala asli: True


**Interpretasi.** Data latih tepat memiliki mean 0 dan std 1. Data uji **tidak** tepat 0/1 — ini justru benar, karena data uji diskalakan memakai statistik data latih, persis seperti yang akan terjadi pada data baru di dunia nyata.

## 5. Hyperparameter vs Parameter Hasil Belajar

| | Hyperparameter | Parameter terlatih |
|---|---|---|
| Ditentukan oleh | kita (sebelum `fit`) | algoritma (saat `fit`) |
| Contoh | `C`, `max_iter`, `n_neighbors`, `alpha` | `coef_`, `intercept_`, `cluster_centers_` |
| Cara akses | argumen konstruktor, `get_params()`, `set_params()` | atribut berakhiran `_` |

Konvensi ini membuat estimator mudah di-*tuning* otomatis (misalnya oleh `GridSearchCV`, yang memakai `set_params()` di belakang layar) dan mudah di-*clone* (`sklearn.base.clone` menyalin hyperparameter tanpa menyalin hasil belajar).

In [7]:
from sklearn.base import clone

print('Beberapa hyperparameter LogisticRegression:')
params = clf.get_params()
print({k: params[k] for k in ['C', 'max_iter', 'solver', 'tol']})

# Ubah hyperparameter dengan set_params (mengembalikan objek yang sama)
clf.set_params(C=0.1)
print('\nC setelah set_params:', clf.get_params()['C'])

# clone: salin hyperparameter, TANPA parameter terlatih
clf_copy = clone(clf)
print('Apakah clone punya coef_? ', hasattr(clf_copy, 'coef_'))
print('Apakah model asli punya coef_?', hasattr(clf, 'coef_'))

Beberapa hyperparameter LogisticRegression:
{'C': 1.0, 'max_iter': 1000, 'solver': 'lbfgs', 'tol': 0.0001}

C setelah set_params: 0.1
Apakah clone punya coef_?  False
Apakah model asli punya coef_? True


## 6. Format Data dan Dataset Bawaan

scikit-learn mengharapkan:
- **`X`**: array 2D berukuran `(n_samples, n_features)` — satu baris = satu observasi, satu kolom = satu fitur.
- **`y`**: array 1D berukuran `(n_samples,)`.

Kesalahan yang sangat umum adalah memberikan satu fitur sebagai array 1D. Solusinya `reshape(-1, 1)`.

Dataset bawaan tersedia dalam dua jenis:
- `load_*` → dataset kecil yang ikut terpasang (iris, wine, digits, breast_cancer, diabetes)
- `make_*` → generator data sintetis (`make_classification`, `make_regression`, `make_blobs`, `make_moons`, …)
- `fetch_*` → dataset lebih besar yang diunduh dari internet (mis. California Housing)

In [8]:
from sklearn.datasets import make_classification

print('Bentuk iris.data  :', iris.data.shape, '-> (n_samples, n_features)')
print('Bentuk iris.target:', iris.target.shape)
print('Nama fitur        :', iris.feature_names)

# Satu fitur harus berbentuk 2D
x_1d = np.array([1, 2, 3, 4])
print('\n1D :', x_1d.shape, '-> reshape(-1, 1) ->', x_1d.reshape(-1, 1).shape)

# as_frame=True mengembalikan DataFrame pandas
iris_df = load_iris(as_frame=True).frame
display(iris_df.head())

# Data sintetis
X_syn, y_syn = make_classification(n_samples=200, n_features=5, n_informative=3, random_state=0)
print('Data sintetis:', X_syn.shape, '| proporsi kelas:', np.bincount(y_syn) / len(y_syn))

Bentuk iris.data  : (150, 4) -> (n_samples, n_features)
Bentuk iris.target: (150,)
Nama fitur        : ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']

1D : (4,) -> reshape(-1, 1) -> (4, 1)


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0


Data sintetis: (200, 5) | proporsi kelas: [0.495 0.505]


## 7. Pipeline — Merangkai Langkah-Langkah

### Teori
`Pipeline` merangkai beberapa transformer dan satu estimator akhir menjadi **satu estimator baru**. Jika langkahnya $T_1, T_2, \dots, T_m$ lalu model $f$, maka prediksi pipeline adalah komposisi fungsi:

$$\hat y = f\big(T_m(\cdots T_2(T_1(\mathbf{x}))\cdots)\big)$$

Saat `pipe.fit(X, y)`: setiap transformer menjalankan `fit_transform` berurutan, lalu estimator terakhir menjalankan `fit`. Saat `pipe.predict(X)`: setiap transformer hanya menjalankan `transform`. Keuntungannya:
1. **Mencegah data leakage** secara otomatis (terutama di dalam cross-validation).
2. Kode lebih ringkas, satu objek bisa disimpan/di-*deploy*.
3. Hyperparameter semua langkah bisa di-tuning bersama dengan notasi `namalangkah__parameter`.

In [9]:
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.decomposition import PCA
from sklearn import set_config

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=2)),
    ('clf', LogisticRegression(max_iter=1000)),
])
pipe.fit(X_train, y_train)
print('Akurasi pipeline: %.3f' % pipe.score(X_test, y_test))

# Akses langkah tertentu
print('Rasio varians PCA:', pipe.named_steps['pca'].explained_variance_ratio_.round(3))

# Mengubah hyperparameter langkah di dalam pipeline: <nama>__<param>
pipe.set_params(pca__n_components=3, clf__C=10)
pipe.fit(X_train, y_train)
print('Akurasi setelah set_params: %.3f' % pipe.score(X_test, y_test))

# make_pipeline memberi nama langkah otomatis
print(make_pipeline(StandardScaler(), LogisticRegression()).steps)

set_config(display='diagram')
pipe

Akurasi pipeline: 0.933
Rasio varians PCA: [0.741 0.216]
Akurasi setelah set_params: 1.000
[('standardscaler', StandardScaler()), ('logisticregression', LogisticRegression())]


Pipeline(steps=[('scaler', StandardScaler()), ('pca', PCA(n_components=3)),
                ('clf', LogisticRegression(C=10, max_iter=1000))])

**Interpretasi.** Dengan 2 komponen PCA, dua komponen pertama menjelaskan sekitar 74% + 22% ≈ 96% variasi data iris yang sudah distandardisasi, dan akurasi uji 0,933. Setelah `pca__n_components=3` dan `clf__C=10`, akurasi naik menjadi 1,000. Yang penting di sini bukan angkanya (data uji hanya 30 sampel), tetapi bahwa seluruh rangkaian langkah dapat diubah dan dilatih ulang sebagai **satu objek**.

## 8. Evaluasi Dasar — Train/Test Split dan Cross-Validation

### Teori
Tujuan ML adalah **generalisasi**: performa baik pada data yang belum pernah dilihat. Karena itu data dibagi menjadi data latih dan data uji. Namun satu kali pembagian bisa "beruntung" atau "sial". **k-fold cross-validation** membagi data menjadi $k$ bagian; model dilatih $k$ kali, setiap kali satu bagian dipakai sebagai validasi:

$$\text{CV}_{(k)} = \frac{1}{k}\sum_{j=1}^{k} \text{skor}_j$$

Rata-rata skor memberi estimasi performa yang lebih stabil, dan simpangan bakunya menunjukkan seberapa sensitif model terhadap pembagian data.

In [10]:
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score, classification_report

pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
scores = cross_val_score(pipe, iris.data, iris.target, cv=5)
print('Skor 5-fold CV :', scores.round(3))
print('Rata-rata ± std: %.3f ± %.3f' % (scores.mean(), scores.std()))

pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)
print('\nAkurasi data uji: %.3f' % accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=iris.target_names))

Skor 5-fold CV : [0.967 1.    0.933 0.9   1.   ]
Rata-rata ± std: 0.960 ± 0.039

Akurasi data uji: 0.967
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       0.91      1.00      0.95        10
   virginica       1.00      0.90      0.95        10

    accuracy                           0.97        30
   macro avg       0.97      0.97      0.97        30
weighted avg       0.97      0.97      0.97        30



**Interpretasi.** Skor per-fold berkisar 0,90–1,00 dengan rata-rata 0,960 ± 0,039. Variasi antar-fold ini menunjukkan mengapa satu kali split saja kurang dapat diandalkan. Pada laporan klasifikasi, *setosa* diprediksi sempurna (kelas ini terpisah linear dari dua kelas lain), sedangkan satu sampel *virginica* tertukar menjadi *versicolor* — sehingga recall *virginica* 0,90 dan precision *versicolor* 0,91.

## Kesimpulan Bab 1

- Semua objek scikit-learn berbagi pola yang sama: **konstruksi → `fit()` → `predict()`/`transform()`/`score()`**.
- **Estimator** belajar dari data; **predictor** menghasilkan prediksi; **transformer** mengubah data.
- Hyperparameter diatur lewat konstruktor/`set_params()`, sedangkan hasil belajar disimpan di atribut berakhiran `_`.
- Data selalu berbentuk matriks `X (n_samples, n_features)` dan vektor `y (n_samples,)`.
- `fit` hanya pada data latih; data uji cukup `transform`/`predict` agar tidak terjadi *data leakage*. **Pipeline** menjamin aturan ini secara otomatis.
- Cross-validation memberi estimasi performa yang lebih andal daripada satu kali train/test split.

Konsep-konsep ini menjadi fondasi untuk semua bab berikutnya.